In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [4]:
df = pd.read_csv("../../data/raw/Gold_market_10y.csv")
df.head()

,Date,Close,High,Low,Open,Volume,Adj Close,Daily_Return,MA_20,MA_50,MA_200,Volatility_20,Year,Month,Day_of_Week,Quarter
0,2016-01-29,106.949997,107.000000,106.260002,106.610001,8098700,106.949997,NaN,NaN,NaN,NaN,NaN,2016,1,4,1
1,2016-02-01,108.050003,108.150002,107.529999,107.540001,10471800,108.050003,1.028524,NaN,NaN,NaN,NaN,2016,2,0,1
2,2016-02-02,108.089996,108.180000,107.349998,107.919998,6656000,108.089996,0.037014,NaN,NaN,NaN,NaN,2016,2,1,1
3,2016-02-03,109.250000,109.580002,107.900002,107.910004,15785200,109.250000,1.073183,NaN,NaN,NaN,NaN,2016,2,2,1
4,2016-02-04,110.570000,110.699997,109.919998,110.449997,13213700,110.570000,1.208238,NaN,NaN,NaN,NaN,2016,2,3,1


In [5]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

df = df.dropna()

df.head()

,Date,Close,High,Low,Open,Volume,Adj Close,Daily_Return,MA_20,MA_50,MA_200,Volatility_20,Year,Month,Day_of_Week,Quarter
199,2016-11-10,119.750000,121.540001,119.379997,121.519997,22268000,119.750000,-1.488975,121.482500,123.4616,121.95685,0.702074,2016,11,3,4
200,2016-11-11,117.099998,119.690002,116.230003,119.650002,20381800,117.099998,-2.212945,121.369500,123.2978,122.00760,0.851717,2016,11,4,4
201,2016-11-14,116.110001,117.139999,115.489998,116.120003,20729400,116.110001,-0.845429,121.191000,123.0886,122.04790,0.863252,2016,11,0,4
202,2016-11-15,117.120003,117.239998,116.290001,116.459999,9348600,117.120003,0.869867,121.026001,122.8568,122.09305,0.876728,2016,11,1,4
203,2016-11-16,116.769997,117.269997,116.580002,117.269997,5486100,116.769997,-0.298844,120.809000,122.6268,122.13065,0.861202,2016,11,2,4


In [6]:
# Returns
df['7_day_return'] = df['Adj Close'].pct_change(7)
df['30_day_return'] = df['Adj Close'].pct_change(30)

# Trend features
df['MA_diff_short'] = df['MA_20'] - df['MA_50']
df['MA_diff_long'] = df['MA_50'] - df['MA_200']

# Volume feature
df['Volume_change'] = df['Volume'].pct_change()

df = df.dropna()

df.head()

,Date,Close,High,Low,Open,Volume,Adj Close,Daily_Return,MA_20,MA_50,...,Volatility_20,Year,Month,Day_of_Week,Quarter,7_day_return,30_day_return,MA_diff_short,MA_diff_long,Volume_change
229,2016-12-23,107.930000,108.250000,107.800003,107.839996,5012200,107.930000,0.316018,110.225500,115.7912,...,0.719376,2016,12,4,4,-0.008270,-0.098706,-5.565700,-5.54240,0.014041
230,2016-12-27,108.559998,108.669998,108.239998,108.610001,3685500,108.559998,0.583709,109.963499,115.5752,...,0.682376,2016,12,1,4,0.011366,-0.072929,-5.611701,-5.71185,-0.264694
231,2016-12-28,108.860001,108.910004,108.290001,108.400002,5091200,108.860001,0.276348,109.743000,115.3588,...,0.689208,2016,12,2,4,0.007497,-0.062441,-5.615800,-5.88275,0.381414
232,2016-12-29,110.290001,110.529999,109.160004,109.230003,7563900,110.290001,1.313614,109.670000,115.1562,...,0.712255,2016,12,3,4,0.015655,-0.058316,-5.486200,-6.03385,0.485681
233,2016-12-30,109.610001,110.620003,109.529999,110.379997,8873700,109.610001,-0.616557,109.573500,114.9262,...,0.722576,2016,12,4,4,0.016885,-0.061317,-5.352700,-6.21125,0.173165


In [7]:
vol_threshold = df['Volatility_20'].median()

def create_signal(row):
    if (row['7_day_return'] > 0.02) and (row['Volatility_20'] < vol_threshold):
        return 2  # BUY
    elif (row['7_day_return'] < -0.02):
        return 0  # SELL
    else:
        return 1  # HOLD

df['Signal'] = df.apply(create_signal, axis=1)

df[['7_day_return', 'Volatility_20', 'Signal']].head()

,7_day_return,Volatility_20,Signal
229,-0.008270,0.719376,1
230,0.011366,0.682376,1
231,0.007497,0.689208,1
232,0.015655,0.712255,1
233,0.016885,0.722576,1


In [8]:
features = [
    'Daily_Return',
    '7_day_return',
    '30_day_return',
    'MA_diff_short',
    'MA_diff_long',
    'Volatility_20',
    'Volume_change'
]

X = df[features]
y = df['Signal']

print(X.shape, y.shape)

(2282, 7) (2282,)


In [9]:
split = int(len(df) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape, X_test.shape)

(1825, 7) (457, 7)


In [10]:
model = GradientBoostingClassifier()
model.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [11]:
y_pred = model.predict(X_test)

y_pred[:10]

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 1])